# ***Download and load pre-trained GloVe and FastText embeddings, then write Python code to convert words into vectors and handle out-of-vocabulary (OOV) words.***
# ***Compute similarity between word pairs and perform analogy tasks (e.g., man : king :: woman : ?) using both models.***


In [1]:
!pip install gensim numpy -q

import numpy as np
from gensim.models import KeyedVectors, Word2Vec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.3 MB/s eta 0:00:00


# ***Demo Vectors (for offline fallback)***

In [2]:
def build_demo_glove():
    dim = 50
    # dims 0-4 encode semantic axes: royalty, gender, animal, geography, sentiment
    def vec(royalty, gender, animal, geo, sentiment, seed):
        np.random.seed(seed)
        v = np.random.normal(0, 0.05, dim).astype(np.float32)
        v[0], v[1], v[2], v[3], v[4] = royalty, gender, animal, geo, sentiment
        return v

    words = {
        "king"    : vec( 0.9,  0.8, -0.1,  0.0,  0.0,  1),
        "queen"   : vec( 0.9, -0.8, -0.1,  0.0,  0.0,  2),
        "prince"  : vec( 0.7,  0.7, -0.1,  0.0,  0.1,  3),
        "princess": vec( 0.7, -0.7, -0.1,  0.0,  0.1,  4),
        "man"     : vec( 0.0,  0.9,  0.0,  0.0,  0.0,  5),
        "woman"   : vec( 0.0, -0.9,  0.0,  0.0,  0.0,  6),
        "boy"     : vec( 0.0,  0.7,  0.0,  0.0,  0.2,  7),
        "girl"    : vec( 0.0, -0.7,  0.0,  0.0,  0.2,  8),
        "cat"     : vec(-0.2,  0.0,  0.9,  0.0,  0.0,  9),
        "dog"     : vec(-0.2,  0.0,  0.8,  0.0,  0.1, 10),
        "lion"    : vec(-0.1,  0.3,  0.9,  0.0, -0.1, 11),
        "tiger"   : vec(-0.1,  0.1,  0.85, 0.0, -0.1, 12),
        "paris"   : vec( 0.0,  0.0, -0.1,  0.9,  0.1, 13),
        "france"  : vec( 0.0,  0.0, -0.1,  0.8,  0.1, 14),
        "berlin"  : vec( 0.0,  0.0, -0.1,  0.85,-0.1, 15),
        "germany" : vec( 0.0,  0.0, -0.1,  0.75,-0.1, 16),
        "london"  : vec( 0.0,  0.0, -0.1,  0.8,  0.0, 17),
        "england" : vec( 0.0,  0.0, -0.1,  0.7,  0.0, 18),
        "good"    : vec( 0.0,  0.0,  0.0,  0.0,  0.9, 19),
        "better"  : vec( 0.0,  0.0,  0.0,  0.0,  0.95,20),
        "bad"     : vec( 0.0,  0.0,  0.0,  0.0, -0.9, 21),
        "worse"   : vec( 0.0,  0.0,  0.0,  0.0, -0.95,22),
        "banana"  : vec(-0.5, -0.2,  0.0,  0.0,  0.3, 23),
        "bird"    : vec(-0.1,  0.0,  0.7,  0.0,  0.1, 24),
        "fish"    : vec(-0.1,  0.0,  0.6,  0.0,  0.0, 25),
    }
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(list(words.keys()), list(words.values()))
    return kv


def build_demo_fasttext():
    # using Word2Vec as stand-in (gensim doesn't expose FastText subword table via KeyedVectors)
    sentences = [
        "the cat sat on the mat".split(),
        "dogs are loyal animals".split(),
        "the king rules the kingdom".split(),
        "the queen rules beside the king".split(),
        "a man walked into the forest".split(),
        "the woman followed the man".split(),
        "the lion is the king of the jungle".split(),
        "animals live in different habitats".split(),
        "good better bad worse".split(),
        "paris is the capital of france".split(),
        "berlin is the capital of germany".split(),
        "man king woman queen prince princess".split(),
    ]
    m = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=1, epochs=300, seed=42)
    return m.wv


# ***Load GloVe***

In [3]:
def load_glove():
    try:
        import gensim.downloader as api
        m = api.load("glove-wiki-gigaword-50")
        return m, False
    except Exception:
        return build_demo_glove(), True

glove, is_demo = load_glove()
mode = "DEMO" if is_demo else "FULL"
print(f"GloVe [{mode}] | vocab: {len(glove):,} | dims: {glove.vector_size}")

[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe [FULL] | vocab: 400,000 | dims: 50


#***Convert Words to Vectors***

In [4]:
print(f"{'word':<10}  first 8 values")
print("-" * 55)
for word in ['king', 'queen', 'man', 'woman', 'cat']:
    vec = glove[word]
    vals = ', '.join([f'{v:.3f}' for v in vec[:8]])
    print(f"{word:<10}  [{vals} ...]")

word        first 8 values
-------------------------------------------------------
king        [0.505, 0.686, -0.595, -0.023, 0.600, -0.135, -0.088, 0.474 ...]
queen       [0.379, 1.823, -1.265, -0.104, 0.358, 0.600, -0.175, 0.838 ...]
man         [-0.094, 0.430, -0.172, -0.455, 1.645, 0.403, -0.373, 0.251 ...]
woman       [-0.182, 0.648, -0.582, -0.495, 1.541, 1.345, -0.433, 0.581 ...]
cat         [0.453, -0.501, -0.537, -0.016, 0.222, 0.546, -0.673, -0.689 ...]


# ***GloVe OOV Handling***

In [5]:
def get_glove_vec(model, word):
    w = word.lower()
    return model[w] if w in model else None


test_words = ['king', 'smartphone', 'ChatGPT', 'dog', 'xyzabc123']

print("GloVe OOV test:")
for w in test_words:
    v = get_glove_vec(glove, w)
    status = f"found  (v[0]={v[0]:.4f})" if v is not None else "OOV — no vector"
    print(f"  {w:<16}  {status}")

GloVe OOV test:
  king              found  (v[0]=0.5045)
  smartphone        found  (v[0]=0.6631)
  ChatGPT           OOV — no vector
  dog               found  (v[0]=0.1101)
  xyzabc123         OOV — no vector


# ***GloVe Word Similarity***

In [6]:
pairs = [
    ('king',  'queen'),
    ('man',   'woman'),
    ('cat',   'dog'),
    ('paris', 'france'),
    ('good',  'bad'),
    ('king',  'banana'),
]

print(f"  {'pair':<22}  {'similarity':>10}")
print(f"  {'-'*22}  {'-'*10}")
for w1, w2 in pairs:
    try:
        s = glove.similarity(w1, w2)
        print(f"  {w1+' / '+w2:<22}  {s:>10.4f}")
    except KeyError as e:
        print(f"  {w1+' / '+w2:<22}  not found: {e}")

  pair                    similarity
  ----------------------  ----------
  king / queen                0.7839
  man / woman                 0.8860
  cat / dog                   0.9218
  paris / france              0.8025
  good / bad                  0.7965
  king / banana               0.2207


# ***GloVe Analogy Tasks***

In [7]:
def solve_analogy(model, a, b, c, tag=""):
    label = f"[{tag}] " if tag else ""
    print(f"  {label}'{a}' : '{b}'  ::  '{c}' : ?")
    try:
        results = model.most_similar(positive=[b, c], negative=[a], topn=3)
        for rank, (word, score) in enumerate(results, 1):
            print(f"    {rank}. {word:<14}  {score:.4f}")
    except KeyError as e:
        print(f"    not found: {e}")
    print()


analogies = [
    ('man',   'king',   'woman'),   # expected: queen
    ('paris', 'france', 'berlin'),  # expected: germany
    ('good',  'better', 'bad'),     # expected: worse
    ('king',  'prince', 'queen'),   # expected: princess
]

for a, b, c in analogies:
    solve_analogy(glove, a, b, c, "GloVe")

  [GloVe] 'man' : 'king'  ::  'woman' : ?
    1. queen           0.8524
    2. throne          0.7664
    3. prince          0.7592

  [GloVe] 'paris' : 'france'  ::  'berlin' : ?
    1. germany         0.9212
    2. denmark         0.8076
    3. poland          0.7936

  [GloVe] 'good' : 'better'  ::  'bad' : ?
    1. worse           0.9011
    2. too             0.8324
    3. unfortunately   0.8224

  [GloVe] 'king' : 'prince'  ::  'queen' : ?
    1. princess        0.8546
    2. duchess         0.7672
    3. victoria        0.7387



# ***Load FastText***

In [ ]:
def load_fasttext():
    try:
        import gensim.downloader as api
        m = api.load("fasttext-wiki-news-subwords-300")
        return m, False
    except Exception:
        return build_demo_fasttext(), True

ft, ft_demo = load_fasttext()
mode = "DEMO" if ft_demo else "FULL"
print(f"FastText [{mode}] | vocab: {len(ft):,} | dims: {ft.vector_size}")

# ***FastText OOV via Character N-grams***

In [ ]:
def char_ngrams(word, min_n=3, max_n=6):
    padded = f"<{word}>"
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(padded) - n + 1):
            ngrams.append(padded[i:i+n])
    return ngrams


def get_fasttext_vec(model, word):
    w = word.lower()
    if w in model:
        return model[w], "exact match"
    ngrams = char_ngrams(w)
    known = [ng for ng in ngrams if ng in model]
    if known:
        vecs = np.array([model[ng] for ng in known])
        return vecs.mean(axis=0).astype(np.float32), f"n-gram avg ({len(known)}/{len(ngrams)} found)"
    return np.zeros(model.vector_size, dtype=np.float32), "zero fallback"


# show what n-grams look like
print("n-grams for 'playing':")
print(" ", char_ngrams("playing"))
print()

# compare GloVe vs FastText OOV coverage
oov_words = ['king', 'animals', 'unhappiness', 'playfulness',
             'ChatGPT', 'xyzabc123', 'running', 'walked']

print(f"  {'word':<20}  {'GloVe':<14}  FastText")
print(f"  {'-'*20}  {'-'*12}  {'-'*30}")
for w in oov_words:
    g = "found" if get_glove_vec(glove, w) is not None else "OOV"
    _, f_method = get_fasttext_vec(ft, w)
    print(f"  {w:<20}  {g:<14}  {f_method}")

# ***Similarity: GloVe vs FastText***

In [ ]:
def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))


print(f"  {'pair':<22}  {'GloVe':>8}  {'FastText':>8}")
print(f"  {'-'*22}  {'-'*8}  {'-'*8}")
for w1, w2 in pairs:
    try:
        g = glove.similarity(w1, w2)
    except KeyError:
        g = None
    v1_vec, _ = get_fasttext_vec(ft, w1)
    v2_vec, _ = get_fasttext_vec(ft, w2)
    f_sim = cosine_sim(v1_vec, v2_vec)
    g_str = f"{g:8.4f}" if g is not None else "     N/A"
    print(f"  {w1+' / '+w2:<22}  {g_str}  {f_sim:8.4f}")

# ***FastText Analogies***

In [ ]:
for a, b, c in analogies:
    solve_analogy(ft, a, b, c, "FastText")